Import Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

import joblib

2: Load Dataset

In [ ]:
df = pd.read_csv("../data/processed/heart_clean.csv")

df.head()

3: Separate Features and Target

In [ ]:
X = df.drop("target", axis=1)

y = df["target"]

4: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

5: Feature Lists

In [ ]:
numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

6: Create Preprocessing Pipeline

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

7: Logistic Regression Pipeline

In [ ]:
lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

8: Train Logistic Regression

In [ ]:
lr_pipeline.fit(X_train, y_train)

9: Predictions

In [ ]:
y_pred_lr = lr_pipeline.predict(X_test)

y_prob_lr = lr_pipeline.predict_proba(X_test)[:,1]

10: Logistic Regression Evaluation

In [ ]:
print("Accuracy :", accuracy_score(y_test,y_pred_lr))

print("Precision :", precision_score(y_test,y_pred_lr))

print("Recall :", recall_score(y_test,y_pred_lr))

print("F1 :", f1_score(y_test,y_pred_lr))

print("ROC-AUC :", roc_auc_score(y_test,y_prob_lr))

11: Classification Report

In [ ]:
print(classification_report(
    y_test,
    y_pred_lr
))

12: Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    lr_pipeline,
    X_test,
    y_test,
    cmap="Blues"
)

plt.title("Logistic Regression")

plt.show()

13: ROC Curve

In [ ]:
RocCurveDisplay.from_estimator(
    lr_pipeline,
    X_test,
    y_test
)

plt.title("ROC Curve")

plt.show()

14: Random Forest Pipeline

In [ ]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                random_state=42
            )
        )
    ]
)

15: Hyperparameter Tuning

In [ ]:
param_grid = {

    "classifier__n_estimators":[100,200],

    "classifier__max_depth":[5,10,None],

    "classifier__min_samples_split":[2,5]
}

grid_search = GridSearchCV(

    rf_pipeline,

    param_grid,

    cv=5,

    scoring="roc_auc"

)

grid_search.fit(

    X_train,

    y_train

)

16: Best Model

In [ ]:
best_model = grid_search.best_estimator_

print(grid_search.best_params_)

17: Predictions

In [ ]:
y_pred_rf = best_model.predict(X_test)

y_prob_rf = best_model.predict_proba(X_test)[:,1]

18: Evaluation

In [ ]:
print("Accuracy :",accuracy_score(y_test,y_pred_rf))

print("Precision :",precision_score(y_test,y_pred_rf))

print("Recall :",recall_score(y_test,y_pred_rf))

print("F1 :",f1_score(y_test,y_pred_rf))

print("ROC-AUC :",roc_auc_score(y_test,y_prob_rf))

19: Classification Report

In [ ]:
print(classification_report(

    y_test,

    y_pred_rf

))

20: Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_estimator(

    best_model,

    X_test,

    y_test,

    cmap="Greens"

)

plt.title("Random Forest")

plt.show()

21: ROC Curve

In [ ]:
RocCurveDisplay.from_estimator(

    best_model,

    X_test,

    y_test

)

plt.title("Random Forest ROC")

plt.show()

22: Cross Validation

In [ ]:
scores = cross_val_score(

    best_model,

    X,

    y,

    cv=5,

    scoring="accuracy"

)

print("Cross Validation Accuracy")

print(scores)

print(scores.mean())

23: Model Comparison

In [ ]:
comparison = pd.DataFrame({

    "Model":[

        "Logistic Regression",

        "Random Forest"

    ],

    "Accuracy":[

        accuracy_score(y_test,y_pred_lr),

        accuracy_score(y_test,y_pred_rf)

    ],

    "Precision":[

        precision_score(y_test,y_pred_lr),

        precision_score(y_test,y_pred_rf)

    ],

    "Recall":[

        recall_score(y_test,y_pred_lr),

        recall_score(y_test,y_pred_rf)

    ],

    "F1":[

        f1_score(y_test,y_pred_lr),

        f1_score(y_test,y_pred_rf)

    ],

    "ROC-AUC":[

        roc_auc_score(y_test,y_prob_lr),

        roc_auc_score(y_test,y_prob_rf)

    ]

})

comparison

24: Save Best Model

In [ ]:
import os

os.makedirs("../models",exist_ok=True)

joblib.dump(

    best_model,

    "../models/best_model.pkl"

)

print("Model Saved Successfully")